# PCA and UMAP on Red Wine (Wine Quality)

The Wine Quality dataset (Cortez et al., 2009) describes 1,599 Portuguese red wines (vinho verde) by 11 physicochemical measurements (acidity, sugar, chlorides, sulfur, density, pH, sulphates, alcohol) and a quality score given by tasters, from 3 to 8. On this real dataset, you reduce the dimension and you interpret the result:

1. Should we scale?
2. Choosing k
3. Reading the axes: the extreme wines
4. Can quality be read on the projection?
5. Non-linear projections: UMAP and t-SNE

The starting cell loads the data into `df` (the score in the `quality` column), puts the 11 measurements into `X` and their standardized version into `X_scaled`. Parts 1 and 2 are independent; parts 3 to 5 follow on from one another (part 3 computes the `X_pca` projection, part 4 the quality groups that part 5 reuses). No GPU needed.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh wine-quality

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# The CSV's first column numbers the wines: it's the index, not a measurement
df = pd.read_csv("wine-quality/wineQualityReds.csv", index_col=0)
print(df.shape)
display(df.head(10))

X = df.drop(columns="quality")
X_scaled = StandardScaler().fit_transform(X)
print("X:", X.shape, "X_scaled:", X_scaled.shape)

## 1. Should we scale?

scikit-learn's PCA centers the data without scaling it. Fit a `PCA` that keeps every axis on `X`, then another one on `X_scaled`. For each, print the variance ratio of the first component, PC1, and its largest coefficients in absolute value (`components_[0]`, with the name of each measurement). Also print the standard deviation of each measurement of `X`.

- Without scaling, which measurement makes PC1? Why that one?
- Should we scale on this dataset? What does PC1 become once the measurements are scaled?

**Useful documentation**

- [`sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [`pandas.Series.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_values.html)
- [`pandas.DataFrame.std`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.std.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `PCA().fit(data)` keeps every axis; `pca.explained_variance_ratio_[0]` is PC1's variance ratio
- `pd.Series(pca.components_[0], index=X.columns)`: PC1's coefficients, with the name of each measurement
- `.sort_values(key=abs, ascending=False)` sorts them by decreasing absolute value
- `X.std()`: the standard deviation of each column, in the measurement's unit
- expected: without scaling, PC1 keeps 95% of the variance, and its largest coefficient is 0.98; with scaling, PC1 keeps 28%

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for name, data in [("not scaled", X), ("scaled", X_scaled)]:
  pca = ...
  coefficients = pd.Series(..., index=X.columns)
  print(f"{name}: PC1 keeps {...:.0%} of the variance")
  print(coefficients.sort_values(key=abs, ascending=False).round(2).head(4))
  print()
print("Standard deviation of each measurement:")
print(X.std().sort_values(...).round(4))
```

### Solution

In [ ]:
for name, data in [("not scaled", X), ("scaled", X_scaled)]:
  pca = PCA().fit(data)
  coefficients = pd.Series(pca.components_[0], index=X.columns)
  print(f"{name}: PC1 keeps {pca.explained_variance_ratio_[0]:.0%} of the variance")
  print(coefficients.sort_values(key=abs, ascending=False).round(2).head(4))
  print()
print("Standard deviation of each measurement:")
print(X.std().sort_values(ascending=False).round(4))
# Without scaling, PC1 keeps 95% of the variance and is almost the total sulfur
# dioxide alone (coefficient 0.98): in mg/dm³, its standard deviation (33) dwarfs the
# other measurements' (less than 2, except the free sulfur, 10). PC1 follows the
# measurement's unit, not what it says about the wines.
# The 11 measurements have different units (g/dm³, mg/dm³, % of alcohol, pH): we must
# scale (standardized PCA). PC1 then keeps only 28% of the variance, and mixes several
# measurements (fixed acidity, citric acid, pH, density).

## 2. Choosing k

As on the "Choosing k and Going to Production" slide:

- fit on `X_scaled` a `PCA()` that keeps the 11 axes, and plot the cumulative explained variance against k;
- print the share of the variance the first 2 axes keep, those of a scatter plot;
- find the smallest k that keeps 80% and 95% of the variance, and check that `PCA(0.8)` and `PCA(0.95)` find the same ones (their `n_components_` attribute).

What does a 2D scatter plot hide on this dataset? Why does it take so many axes, when two are enough on Iris (96%)?

**Useful documentation**

- [`numpy.cumsum`](https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html)
- [`numpy.searchsorted`](https://numpy.org/doc/stable/reference/generated/numpy.searchsorted.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `np.cumsum(pca.explained_variance_ratio_)`: the variance kept by the first k axes, for k = 1 to 11; `cumulative[1]` is the first 2's
- `np.searchsorted(cumulative, 0.95) + 1`: the smallest k whose cumulative variance reaches 95% (`searchsorted` returns an index, which starts at 0)
- `plt.axhline(0.95, color="gray", linestyle="--")` marks the threshold on the plot
- expected: 46% for 2 axes; k = 6 for 80%, k = 9 for 95%

A skeleton to copy into the exercise cell, replacing each `...`:

```python
cumulative = np.cumsum(...)
plt.plot(np.arange(1, len(cumulative) + 1), cumulative, marker=".")
plt.axhline(0.95, color="gray", linestyle="--")
plt.xlabel("k")
plt.ylabel("Cumulative explained variance")
plt.show()

print(f"The first 2 axes keep: {...:.0%}")
for share in [0.8, 0.95]:
  k = ...
  n_components = ...
  print(f"{share:.0%}: k = {k}, PCA: {n_components}")
```

### Solution

In [ ]:
cumulative = np.cumsum(PCA().fit(X_scaled).explained_variance_ratio_)
plt.plot(np.arange(1, len(cumulative) + 1), cumulative, marker=".")
plt.axhline(0.95, color="gray", linestyle="--")
plt.xlabel("k")
plt.ylabel("Cumulative explained variance")
plt.show()

print(f"The first 2 axes keep: {cumulative[1]:.0%}")
for share in [0.8, 0.95]:
  k = np.searchsorted(cumulative, share) + 1
  n_components = PCA(share).fit(X_scaled).n_components_
  print(f"{share:.0%}: k = {k}, PCA: {n_components}")
# The first 2 axes keep only 46% of the variance: a 2D plot shows less than half of
# the differences between the wines, and two wines close on the plot can be far apart
# in the 11 measurements. It takes 6 axes for 80%, 9 out of 11 for 95%: the curve rises
# slowly, because the measurements are weakly correlated with one another and the
# variance spreads over many directions. On Iris, three of the four measurements are
# highly correlated, and two axes keep 96%.

## 3. Reading the axes: the extreme wines

- Fit a `PCA(n_components=2)` on `X_scaled` and store the projection in `X_pca` (the next parts use it). Print its coefficients in a `components` `DataFrame`, one row per axis and one column per measurement, and name each axis after its largest coefficients, as on the "Reading a PCA" slide.
- Plot the `X_pca` scatter. A few wines stand out: find the one with the largest coordinate on PC1, and the one with the largest on PC2.
- Explain where they are. A wine's coordinate on an axis is the sum, over the 11 measurements, of its scaled value (its row of `X_scaled`) times the measurement's coefficient: these are the measurements' contributions. Print the largest ones for each of the two wines, then its measurements in `df`, next to their median.

Which measurements put each of these wines so far out? Is it the same kind of wine as the rest of the dataset?

**Useful documentation**

- [`numpy.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)
- [`pandas.DataFrame.iloc`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `pd.DataFrame(pca.components_, index=["PC1", "PC2"], columns=X.columns)`; its transpose, `components.T`, reads better (one row per measurement)
- `np.argmax(X_pca[:, 0])`: the position of the rightmost wine; its number in the CSV is `df.index[i]`
- the contributions of wine `i` to axis `k`: `X_scaled[i] * pca.components_[k]`; their sum is `X_pca[i, k]`
- `df.iloc[i]`: the measurements of the wine at position `i`; `df.median()`: the median of each column
- expected: PC1's largest coefficient is the fixed acidity's (0.49 in absolute value); the rightmost wine is no. 152 (PC1 = 8.7), the highest no. 1435 (PC2 = 6.8)

A skeleton to copy into the exercise cell, replacing each `...`:

```python
pca = PCA(n_components=2)
X_pca = ...
components = pd.DataFrame(..., index=["PC1", "PC2"], columns=X.columns)
display(components.T.round(2))

plt.figure(figsize=(7, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=5, alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

for k in range(2):
  i = ...
  contributions = pd.Series(..., index=X.columns)
  print(f"Wine no. {df.index[i]}: PC{k + 1} = {X_pca[i, k]:.1f}")
  print(contributions.sort_values(key=abs, ascending=False).round(2).head(4))
  display(
    pd.DataFrame({f"Wine no. {df.index[i]}": df.iloc[i], "median": df.median()}).T
  )
```

### Solution

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
components = pd.DataFrame(pca.components_, index=["PC1", "PC2"], columns=X.columns)
display(components.T.round(2))

plt.figure(figsize=(7, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=5, alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

for k in range(2):
  i = np.argmax(X_pca[:, k])
  contributions = pd.Series(X_scaled[i] * pca.components_[k], index=X.columns)
  print(f"Wine no. {df.index[i]}: PC{k + 1} = {X_pca[i, k]:.1f}")
  print(contributions.sort_values(key=abs, ascending=False).round(2).head(4))
  display(
    pd.DataFrame({f"Wine no. {df.index[i]}": df.iloc[i], "median": df.median()}).T
  )
# PC1 opposes acidity (fixed acidity, citric acid, density) to pH: an "acidity" axis.
# PC2 opposes sulfur (total and free sulfur dioxide) to alcohol.
# Wine no. 152 owes its place on PC1 to its chlorides (0.61 g/dm³, against 0.08 for the
# median), its sulphates (2.0 against 0.62) and its citric acid, the highest of the
# dataset, and to its very low pH: an outlier, not a very acidic wine like the other
# wines on the right. No. 1435 owes its place on PC2 to its residual sugar (15.4 g/dm³,
# against 2.2) and its free sulfur: an almost sweet wine, rare in this dataset (no.
# 1436, the same wine duplicated, is at the same place). An extreme point is explained
# by the contributions: a measurement's coefficient isn't enough, the wine must also be
# far from the mean on that measurement.

## 4. Can quality be read on the projection?

- Print the distribution of the quality scores.
- Sort the wines into three groups in a `groups` series: the bad ones (score below 5), the average ones (5 and 6) and the good ones (7 and above), and print the size of each group.
- Complete the `show_projection(ax, Z, title)` function, which plots a 2D projection `Z` on the `ax` plot, each wine colored by its group (the average ones, numerous, in light gray and transparent, under the others), then plot `X_pca` with it. Part 5 reuses it.

Do the three groups separate on the projection? Where do the good wines lean, and which measurement of part 3 explains it? Why doesn't PCA, unsupervised, try to separate quality?

**Useful documentation**

- [`pandas.Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
- [`pandas.cut`](https://pandas.pydata.org/docs/reference/api/pandas.cut.html)

In [ ]:
def show_projection(ax, Z, title):
  # Plots the 2D projection Z on ax, one point per wine, colored by its group
  ...


# Your code here

### Additional hints

The calls and parameters to use:

- `df["quality"].value_counts().sort_index()`: the number of wines per score
- `pd.cut(df["quality"], bins=[0, 4, 6, 10], labels=[...])`: the intervals are closed on the right, (0, 4], (4, 6] and (6, 10]
- in `show_projection`, a loop over the groups, the gray one first so that it's underneath: `mask = (groups == name).to_numpy()`, then `ax.scatter(Z[mask, 0], Z[mask, 1], ...)`
- expected: 63 bad, 1,319 average and 217 good

A skeleton to copy into the exercise cell, replacing each `...`:

```python
print(df["quality"].value_counts().sort_index())
groups = pd.cut(..., bins=[0, 4, 6, 10], labels=["bad", "average", "good"])
print(groups.value_counts())

COLORS = {"average": "lightgray", "bad": "tab:red", "good": "tab:green"}


def show_projection(ax, Z, title):
  for name, color in COLORS.items():
    mask = ...
    alpha = 0.4 if name == "average" else 0.8
    ax.scatter(..., s=5, color=color, alpha=alpha, label=name)
  ax.set_title(title)
  ax.legend()


fig, ax = plt.subplots(figsize=(7, 6))
show_projection(ax, X_pca, "PCA")
plt.show()
```

### Solution

In [ ]:
print(df["quality"].value_counts().sort_index())
groups = pd.cut(df["quality"], bins=[0, 4, 6, 10], labels=["bad", "average", "good"])
print(groups.value_counts())

COLORS = {"average": "lightgray", "bad": "tab:red", "good": "tab:green"}


def show_projection(ax, Z, title):
  # Plots the 2D projection Z on ax, one point per wine, colored by its group
  for name, color in COLORS.items():
    mask = (groups == name).to_numpy()
    alpha = 0.4 if name == "average" else 0.8
    ax.scatter(Z[mask, 0], Z[mask, 1], s=5, color=color, alpha=alpha, label=name)
  ax.set_title(title)
  ax.legend()


fig, ax = plt.subplots(figsize=(7, 6))
show_projection(ax, X_pca, "PCA")
plt.show()
# No: the three groups overlap. The good wines lean towards the bottom of PC2, the
# alcohol side (a coefficient of -0.39 on PC2, and alcohol is the measurement most
# related to the score); the bad ones slightly towards the left of PC1. But PCA doesn't
# know the score: it looks for the directions of largest variance of the 11
# measurements (acidity, sulfur), which aren't the ones that make quality. The axis of
# largest variance has no reason to be the most useful for a target: to separate known
# classes, a supervised method is needed (LDA, a model).

## 5. Non-linear projections: UMAP and t-SNE

UMAP and t-SNE keep each wine's neighbors rather than the variance. As their best practices advise, try several settings, with `random_state=0` to get the same map on every run:

- project `X_scaled` in 2D with UMAP for 5, 15 and 50 neighbors (`n_neighbors`), then with t-SNE for a perplexity of 5, 30 and 50;
- plot the six projections with `show_projection`, on a grid of 2 × 3 plots.

Does quality separate better than with PCA? What changes with the number of neighbors or the perplexity, and what remains from one map to the next? What can be read in the size of the groups and the distances between them?

**Useful documentation**

- [`umap.UMAP`](https://umap-learn.readthedocs.io/en/latest/api.html#umap.umap_.UMAP)
- [`sklearn.manifold.TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `umap.UMAP(n_neighbors=..., random_state=0, n_jobs=1)` and `TSNE(perplexity=..., random_state=0)` have a `fit_transform` method, like `PCA`; with `random_state`, UMAP computes on a single core, and `n_jobs=1` keeps it from saying so in a warning
- `fig, axes = plt.subplots(2, 3, figsize=(15, 9))`: `axes[0]` is the first row of plots, `axes[1]` the second
- expected: a few seconds per projection (UMAP's first one takes longer: it compiles its code)

A skeleton to copy into the exercise cell, replacing each `...`:

```python
import umap
from sklearn.manifold import TSNE

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, n_neighbors in zip(axes[0], [5, 15, 50]):
  reducer = umap.UMAP(...)
  show_projection(ax, ..., f"UMAP, n_neighbors = {n_neighbors}")
for ax, perplexity in zip(axes[1], [5, 30, 50]):
  reducer = TSNE(...)
  show_projection(ax, ..., f"t-SNE, perplexity = {perplexity}")
plt.show()
```

### Solution

In [ ]:
import umap
from sklearn.manifold import TSNE

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, n_neighbors in zip(axes[0], [5, 15, 50]):
  reducer = umap.UMAP(n_neighbors=n_neighbors, random_state=0, n_jobs=1)
  show_projection(
    ax, reducer.fit_transform(X_scaled), f"UMAP, n_neighbors = {n_neighbors}"
  )
for ax, perplexity in zip(axes[1], [5, 30, 50]):
  reducer = TSNE(perplexity=perplexity, random_state=0)
  show_projection(
    ax, reducer.fit_transform(X_scaled), f"t-SNE, perplexity = {perplexity}"
  )
plt.show()
# Not really: the good wines gather a little more than on the PCA, on the same side of
# each map, but remain mixed with the average ones. UMAP and t-SNE don't know the score
# either: they keep each wine's neighbors in the 11 measurements.
# With 5 neighbors, UMAP crumbles the map into small islands and isolated points (the
# very local structure); with 50, the overall shape shows. t-SNE changes less, but the
# good wines get closer together on it as the perplexity rises. What remains from one
# map to the next, like the side of the good wines, is more reliable than the detail of
# the islands. The size of the groups and the distances between them aren't reliable on
# these maps.

## Conclusion

- Measurements with different units are scaled before PCA: otherwise, the most spread-out one makes PC1 on its own.
- The cumulative explained variance says how many axes to keep, and what a 2D scatter plot hides (here, more than half of the variance).
- The coefficients name the axes; the contributions explain where a point is.
- PCA doesn't look for the target: quality can't be read on its first axes.
- UMAP and t-SNE: several settings, a fixed `random_state`, and neither the size of the groups nor their distances can be read.